In [1]:
from pathlib import Path

import polars as pl
from tqdm.auto import tqdm

In [ ]:
(
    pl
    .read_parquet("/mnt/Fast2T/data/moltbook.parquet")
    # .filter(pl.col("language") == "ENGLISH")
    .filter(pl.col("text").str.len_bytes() > 500)
    .select("text")
    .sample(10000)
    .write_parquet("/mnt/Fast2T/data/detection/moltbook-filtered.parquet")
)

In [ ]:
(
    pl
    .read_parquet("/mnt/Fast2T/data/detection/awiki-filtered.parquet")
    .select("clean_text")
    .with_columns(text=pl.col("clean_text"))
    .select("text")
    .write_parquet("/mnt/Fast2T/data/detection/wiki-filtered.parquet")
)

In [ ]:
pl.read_parquet("/mnt/Fast2T/data/detection/wiki-filtered.parquet")

In [2]:
from concurrent.futures import ThreadPoolExecutor

files = list(Path("/mnt/data-dump/for_enriching/").glob("*.parquet"))


def count_file(f: Path) -> int:
    if f.stat().st_size < 12:
        return 0
    try:
        return (
            pl.scan_parquet(f)
            .select(pl.col("text").str.len_bytes() > 10)
            .sum()
            .collect()
            .item()
        )
    except Exception as e:
        print(f"Skipping {f.name}: {e}")
        return 0


with ThreadPoolExecutor(max_workers=48) as pool:
    counts = list(tqdm(pool.map(count_file, files), total=len(files)))

print(f"Files: {len(files)}")
print(f"Rows to process: {sum(counts):,}")

  0%|          | 0/1498 [00:00<?, ?it/s]

Skipping c00369.parquet: parquet: File out of specification: The file must end with PAR1
Skipping c00261.parquet: parquet: File out of specification: The file must end with PAR1
Files: 1498
Rows to process: 25,111,645


In [10]:
import pycld2 as cld2


def detect_lang(text: str) -> dict:
    try:
        reliable, _, details = cld2.detect(text)
        return {"lang": details[0][1], "lang_reliable": reliable, "lang_confidence": details[0][2]}
    except:
        return {"lang": None, "lang_reliable": False, "lang_confidence": 0}


(
    pl.read_parquet("/mnt/Fast2T/data/moltbook.parquet")
    .filter(pl.col("text").str.len_bytes() > 1000)
    .select("text")
    .with_columns(
        pl.col("text")
        .map_elements(detect_lang, return_dtype=pl.Struct({
            "lang": pl.Utf8,
            "lang_reliable": pl.Boolean,
            "lang_confidence": pl.Int64,
        }))
        .alias("_detect")
    )
    .unnest("_detect")
    .filter(pl.col("lang_reliable") == True)
    .filter(pl.col("lang") == "en")
    .sample(10000)
    .write_parquet("/mnt/Fast2T/data/detection/moltbook-filtered.parquet")
)

In [ ]:
import fasttext
from pqdm.threads import pqdm

model = fasttext.load_model("lid.176.bin")


def detect_lang(text: str, threshold: float = 0.6) -> bool:
    try:
        text = text.replace("\n", " ").strip()
        # skip boilerplate at edges
        n = len(text)
        chunk = text[n // 4: 3 * n // 4]
        if len(chunk) < 200:
            chunk = text
        labels, scores = model.predict(text, k=2)
        if labels[0] != "__label__en":
            return False
        if len(scores) > 1 and scores[1] > 0.15:
            return False  # significant second language detected
        return bool(scores[0] >= threshold)
    except:
        return False


def process(file: Path):
    try:
        (
            pl
            .scan_parquet(file)
            .filter(pl.col("text").str.len_bytes() > 200)
            .with_columns(
                pl.col("text")
                .map_elements(detect_lang, return_dtype=pl.Boolean)
                .alias("is_english")
            )
            .filter(pl.col("is_english") == True)
            .select("id", "url", "text")
            .sink_parquet(
                Path("/home/qr/data/for_scanning/") / file.name,
                compression="zstd",
                compression_level=12
            )
        )
    except:
        print(f"Skipping {file.name}")


files = sorted(list(Path("/mnt/data-dump/for_enriching/").glob("*.parquet")))

pqdm(files, process, n_jobs=32)

QUEUEING TASKS | :   0%|          | 0/1498 [00:00<?, ?it/s]

Skipping c00005.parquet
Skipping c00017.parquet
Skipping c00029.parquet
Skipping c00032.parquet
Skipping c00035.parquet
Skipping c00036.parquet


PROCESSING TASKS | :   0%|          | 0/1498 [00:00<?, ?it/s]